# Section 2.1 - What are AI-Related Products?
Generates: tables/trade_hierarchical.tex, top-10/11-20 landscape tables.
**Run first** - initializes tables/ai-trade-results.tex.


In [96]:
# Initialize LaTeX output file (creates/clears it) - run before 07/08/09
import os
os.makedirs('paper/tables', exist_ok=True)
texfile = "../paper/tables/ai-trade-results.tex"
figfile = "../paper/figures/"
open(texfile, "w").close()

In [97]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import os

In [98]:
# Define paths for output files
texfile = "../paper/tables/ai-trade-results.tex"
figfile = "../paper/figures/"

In [99]:
matlist = pd.read_csv('../data-input/hs10_classification_final_v3.csv')

# matlist = pd.read_csv('hs10_datacenter_relevance.csv')

# Convert to category
matlist['relevance'] = matlist['relevance'].astype('category')

# Or specify order if needed (e.g., for sorting/comparisons)
matlist['relevance'] = pd.Categorical(
    matlist['relevance'], 
    categories=['Low', 'Medium', 'High'], 
    ordered=True
)

matlist['primary_category'] = matlist['primary_category'].astype('category')

matlist.rename(columns={'hs10_code': 'HS10'}, inplace=True)

In [100]:
matlist.head()

,relevance,confidence,primary_category,specific_use,reasoning,HS10,description,naics_code,naics_description
0,Low,100,Not_DC_Related,No application in data center context,Live purebred breeding horses are livestock us...,101210010,"HORSES, LIVE, PUREBRED BREEDING MALE",112920,HORSES AND OTHER EQUINE
1,Low,100,Not_DC_Related,Not applicable to data center construction or ...,Live horses are livestock animals with no conn...,101210020,"HORSES, LIVE, PUREBRED BREEDING FEMALE",112920,HORSES AND OTHER EQUINE
2,Low,100,Not_DC_Related,No application in data center construction or ...,Live horses are livestock animals with no rele...,101290090,"HORSES, LIVE, NESOI",112920,HORSES AND OTHER EQUINE
3,Low,100,Not_DC_Related,Not applicable - livestock animals have no rol...,Live donkeys/asses are livestock animals used ...,101300000,"ASSES, LIVE",112920,HORSES AND OTHER EQUINE
4,Low,100,Not_DC_Related,Not applicable to data center operations,Live cattle are agricultural livestock with no...,102210010,"CATTLE, LIVE, PUREBRED BREEDING MALE, DAIRY",11211X,Description not found for NAICS 11211X


In [101]:
df = pd.read_parquet('../data-input/TOTALdata-current.parquet')

df.rename(columns={'I_COMMODITY': 'HS10'}, inplace=True)

df["HS2"] = df["HS10"].str[0:2]
df["HS4"] = df["HS10"].str[0:4]

df["HS10"] = df["HS10"].astype('int64')

df.time = pd.to_datetime(df.time, format="%Y-%m")

df["imports"] = df["CON_VAL_MO"].astype(float)

df["duty"] = df["CAL_DUT_MO"].astype(float)

df.rename({"I_COMMODITY_SDESC": "short_description"}, axis = 1, inplace= True)

# Remove volatile/special HS2 categories
# These categories have prices that are either:
# - Extremely volatile (oil, precious metals)
# - Lumpy/irregular (aircraft)
# - Unreliable/special (pharmaceuticals, special provisions)
excluded_hs2 = ["27", "71", "98", "99"]
df = df[~df["HS2"].isin(excluded_hs2)]

In [102]:
df.tail()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,HS10,short_description,time,COMM_LVL,HS2,HS4,imports,duty
2726615,TOTAL FOR ALL COUNTRIES,1031361,308288,1605514000,"OYSTERS, SMOKED",2026-01-01,HS10,16,1605,1031361.0,308288.0
2726616,TOTAL FOR ALL COUNTRIES,582408,92228,1605515000,"OYSTERS EXCEPT SMOKED, PREPARED OR PRESERVED",2026-01-01,HS10,16,1605,582408.0,92228.0
2726617,TOTAL FOR ALL COUNTRIES,310800,36176,1605520500,SCALLOP PRODUCTS WITH FISH MEAT; PREP MEALS,2026-01-01,HS10,16,1605,310800.0,36176.0
2726618,TOTAL FOR ALL COUNTRIES,412467,91158,1605526000,"SCALLOPS, PREPARED OR PRESERVED",2026-01-01,HS10,16,1605,412467.0,91158.0
2726619,TOTAL FOR ALL COUNTRIES,368316,37195,1605530500,MUSSEL PRODUCTS WITH FISH MEAT; PREP MEALS,2026-01-01,HS10,16,1605,368316.0,37195.0


In [103]:
# Merge Relevance category from matlist onto df
df = df.merge(matlist[['HS10', 'relevance', 'primary_category', "reasoning"	]], on='HS10', how='left')

In [104]:
df.head()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,HS10,short_description,time,COMM_LVL,HS2,HS4,imports,duty,relevance,primary_category,reasoning
0,TOTAL FOR ALL COUNTRIES,773010,0,602400000,"ROSES, GRAFTED OR NOT",2013-01-01,HS10,06,0602,773010.0,0.0,Low,Not_DC_Related,Roses are ornamental plants used for landscapi...
1,TOTAL FOR ALL COUNTRIES,6177543,0,602902000,"ORCHID PLANTS, LIVE",2013-01-01,HS10,06,0602,6177543.0,0.0,Low,Not_DC_Related,Live orchid plants are ornamental/decorative p...
2,TOTAL FOR ALL COUNTRIES,135786,0,602903010,CHRYSANTHEMUMS WITH SOIL ATTACHED TO ROOTS,2013-01-01,HS10,06,0602,135786.0,0.0,Low,Not_DC_Related,These are live flowering plants (chrysanthemum...
3,TOTAL FOR ALL COUNTRIES,169439,0,602903090,"HERBACEOUS PERENNIALS,WTH SOIL ATTACHED,LIVE,N...",2013-01-01,HS10,06,0602,169439.0,0.0,Low,Not_DC_Related,This product is live herbaceous perennial plan...
4,TOTAL FOR ALL COUNTRIES,2177498,25345,602904000,HERBACEOUS PERENNIALS WTHOUT SOIL ATTACHED NESOI,2013-01-01,HS10,06,0602,2177498.0,25345.0,Low,Not_DC_Related,Live herbaceous perennial plants from nurserie...


In [105]:
# Generate LaTeX table for trade values by relevance (2023 vs 2025)

import os
os.makedirs('paper/tables', exist_ok=True)

# Calculate values for the table
df_reset = df.reset_index()

# Count unique HS10 codes by relevance
high_hs_count = df_reset[df_reset['relevance'] == 'High']['HS10'].nunique()
low_hs_count = df_reset[df_reset['relevance'] == 'Low']['HS10'].nunique()
total_hs_count = df_reset[df_reset['time'].dt.year == 2024]['HS10'].nunique()

# 2023 values
high_2023 = df_reset[(df_reset['time'].dt.year == 2023) & (df_reset['relevance'] == 'High')]['imports'].sum()
low_2023 = df_reset[(df_reset['time'].dt.year == 2023) & (df_reset['relevance'] == 'Low')]['imports'].sum()
total_2023 = df_reset[df_reset['time'].dt.year == 2023]['imports'].sum()

# 2025 values
high_2025 = df_reset[(df_reset['time'].dt.year == 2025) & (df_reset['relevance'] == 'High')]['imports'].sum()
low_2025 = df_reset[(df_reset['time'].dt.year == 2025) & (df_reset['relevance'] == 'Low')]['imports'].sum()
total_2025 = df_reset[df_reset['time'].dt.year == 2025]['imports'].sum()

# Calculate percent increases
high_pct = ((high_2025 - high_2023) / high_2023) * 100
low_pct = ((low_2025 - low_2023) / low_2023) * 100
total_pct = ((total_2025 - total_2023) / total_2023) * 100

# Generate LaTeX table
table_file = "../paper/tables/trade_by_relevance.tex"
with open(table_file, 'w', encoding='utf-8') as f:
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\caption{U.S. Import Values by AI Relevance (2023 vs 2025)}\n")
    f.write("\\label{tab:trade_by_relevance}\n")
    f.write("\\setlength{\\tabcolsep}{4.5mm}\n")
    f.write("\\renewcommand{\\arraystretch}{1.60}\n")
    f.write("\\begin{tabular}{lrrrr}\n")
    f.write("\\toprule\n")
    f.write("Category & \\# HS10 Codes & 2023 (\\$B) & 2025 (\\$B) & Change (\\%) \\\\\n")
    f.write("\\midrule\n")
    f.write(f"High AI Relevance & {high_hs_count} & {high_2023/1e9:.1f} & {high_2025/1e9:.1f} & {high_pct:+.1f} \\\\\n")
    f.write(f"Low AI Relevance & {low_hs_count} & {low_2023/1e9:.1f} & {low_2025/1e9:.1f} & {low_pct:+.1f} \\\\\n")
    f.write("\\midrule\n")
    f.write(f"Total Trade & {total_hs_count} & {total_2023/1e9:.1f} & {total_2025/1e9:.1f} & {total_pct:+.1f} \\\\\n")
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")

print(f"Generated: {table_file}\n")
print("LaTeX Table:")
print("="*80)
with open(table_file, 'r', encoding='utf-8') as f:
    print(f.read())
print("="*80)
print("\nSummary:")
print(f"  High AI Relevance: {high_hs_count} HS10 codes | ${high_2023/1e9:.1f}B (2023) → ${high_2025/1e9:.1f}B (2025) [{high_pct:+.1f}%]")
print(f"  Low AI Relevance:  {low_hs_count} HS10 codes | ${low_2023/1e9:.1f}B (2023) → ${low_2025/1e9:.1f}B (2025) [{low_pct:+.1f}%]")
print(f"  Total Trade:       {total_hs_count} HS10 codes | ${total_2023/1e9:.1f}B (2023) → ${total_2025/1e9:.1f}B (2025) [{total_pct:+.1f}%]")

Generated: ../paper/tables/trade_by_relevance.tex

LaTeX Table:
\begin{table}[htbp]
\centering
\caption{U.S. Import Values by AI Relevance (2023 vs 2025)}
\label{tab:trade_by_relevance}
\setlength{\tabcolsep}{4.5mm}
\renewcommand{\arraystretch}{1.60}
\begin{tabular}{lrrrr}
\toprule
Category & \# HS10 Codes & 2023 (\$B) & 2025 (\$B) & Change (\%) \\
\midrule
High AI Relevance & 645 & 379.0 & 654.0 & +72.6 \\
Low AI Relevance & 15915 & 1834.7 & 1880.9 & +2.5 \\
\midrule
Total Trade & 18364 & 2598.3 & 2883.9 & +11.0 \\
\bottomrule
\end{tabular}
\end{table}


Summary:
  High AI Relevance: 645 HS10 codes | $379.0B (2023) → $654.0B (2025) [+72.6%]
  Low AI Relevance:  15915 HS10 codes | $1834.7B (2023) → $1880.9B (2025) [+2.5%]
  Total Trade:       18364 HS10 codes | $2598.3B (2023) → $2883.9B (2025) [+11.0%]


In [106]:
# Write trade by relevance values to LaTeX file
with open(texfile, 'a') as f:
    # High AI Relevance
    f.write(f'\\def\\highAiHsCodes{{{high_hs_count}}} % Number of HS10 codes classified as high AI relevance\n')
    f.write(f'\\def\\highAiTwentyThree{{{high_2023/1e9:.0f}}} % High AI relevance imports in 2023 ($B)\n')
    f.write(f'\\def\\highAiTwentyFive{{{high_2025/1e9:.0f}}} % High AI relevance imports in 2025 ($B)\n')
    f.write(f'\\def\\highAiPctChange{{{high_pct:.0f}}} % High AI relevance percent change 2023-2025\n')
    
    # Low AI Relevance
    f.write(f'\\def\\lowAiHsCodes{{{low_hs_count}}} % Number of HS10 codes classified as low AI relevance\n')
    f.write(f'\\def\\lowAiTwentyThree{{{low_2023/1e9:.0f}}} % Low AI relevance imports in 2023 ($B)\n')
    f.write(f'\\def\\lowAiTwentyFive{{{low_2025/1e9:.0f}}} % Low AI relevance imports in 2025 ($B)\n')
    f.write(f'\\def\\lowAiPctChange{{{low_pct:.0f}}} % Low AI relevance percent change 2023-2025\n')
    
    # Total Trade
    f.write(f'\\def\\totalTradeHsCodes{{{total_hs_count}}} % Total number of HS10 codes in dataset\n')
    f.write(f'\\def\\totalTradeTwentyThree{{{total_2023/1e9:.0f}}} % Total imports in 2023 ($B)\n')
    f.write(f'\\def\\totalTradeTwentyFive{{{total_2025/1e9:.0f}}} % Total imports in 2025 ($B)\n')
    f.write(f'\\def\\totalTradePctChange{{{total_pct:.0f}}} % Total trade percent change 2023-2025\n')

print(f"\nAppended trade by relevance values to {texfile}")


Appended trade by relevance values to ../paper/tables/ai-trade-results.tex


In [107]:
# Generate LaTeX table for trade values by AI category (High relevance only)

import os
os.makedirs('paper/tables', exist_ok=True)

# Calculate values for the table
df_reset = df.reset_index()

# Get high relevance only
df_high_only = df_reset[df_reset['relevance'] == 'High'].copy()

# Get all categories (excluding Not_DC_Related, Maintenance_Operations, and NaN)
all_cats = [cat for cat in df_high_only['primary_category'].unique() 
            if pd.notna(cat) and cat != 'Not_DC_Related' and cat != 'Maintenance_Operations']

category_data = {}
for cat in all_cats:
    cat_data = df_high_only[df_high_only['primary_category'] == cat]
    
    # Count unique HS10 codes
    hs_count = cat_data['HS10'].nunique()
    
    # 2023 values
    cat_2023 = cat_data[cat_data['time'].dt.year == 2023]['imports'].sum()
    
    # 2025 values
    cat_2025 = cat_data[cat_data['time'].dt.year == 2025]['imports'].sum()
    
    # Calculate percent increase
    cat_pct = ((cat_2025 - cat_2023) / cat_2023) * 100 if cat_2023 > 0 else 0
    
    category_data[cat] = {
        'hs_count': hs_count,
        '2023': cat_2023,
        '2025': cat_2025,
        'pct': cat_pct
    }

# Calculate totals for High AI Relevance
# Sort categories by 2023 dollar value (descending)
all_cats = sorted(category_data.keys(), key=lambda x: category_data[x]['2023'], reverse=True)

# Calculate totals for High AI Relevance
total_high_hs_count = df_high_only['HS10'].nunique()
total_high_2023 = df_high_only[df_high_only['time'].dt.year == 2023]['imports'].sum()
total_high_2025 = df_high_only[df_high_only['time'].dt.year == 2025]['imports'].sum()
total_high_pct = ((total_high_2025 - total_high_2023) / total_high_2023) * 100

# Generate LaTeX table
table_file = "../paper/tables/trade_by_category.tex"
with open(table_file, 'w', encoding='utf-8') as f:
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\caption{U.S. Import Values by AI Category, High AI Relevance (2023 vs 2025)}\n")
    f.write("\\label{tab:trade_by_category}\n")
    f.write("\\setlength{\\tabcolsep}{4.5mm}\n")
    f.write("\\renewcommand{\\arraystretch}{1.60}\n")
    f.write("\\begin{tabular}{lrrrr}\n")
    f.write("\\toprule\n")
    f.write("Category & \\# HS10 Codes & 2023 (\\$B) & 2025 (\\$B) & Change (\\%) \\\\\n")
    f.write("\\midrule\n")
    
    for cat in all_cats:
        cat_name = cat.replace('_', ' ')
        data = category_data[cat]
        f.write(f"{cat_name} & {data['hs_count']} & {data['2023']/1e9:.1f} & {data['2025']/1e9:.1f} & {data['pct']:+.1f} \\\\\n")
    
    f.write("\\midrule\n")
    f.write(f"Total High AI Relevance & {total_high_hs_count} & {total_high_2023/1e9:.1f} & {total_high_2025/1e9:.1f} & {total_high_pct:+.1f} \\\\\n")
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")

print(f"Generated: {table_file}\n")
print("LaTeX Table:")
print("\nSummary by Category (High Relevance):")
with open(table_file, 'r', encoding='utf-8') as f:
    print(f.read())
print("="*80)
print("\nSummary by Category (High Relevance):")
for cat in all_cats:

    data = category_data[cat]
    print(f"\n  Total High AI Relevance: {total_high_hs_count} HS10 codes | ${total_high_2023/1e9:.1f}B (2023) → ${total_high_2025/1e9:.1f}B (2025) [{total_high_pct:+.1f}%]")
    print(f"  {cat_name}: {data['hs_count']} HS10 codes | ${data['2023']/1e9:.1f}B (2023) → ${data['2025']/1e9:.1f}B (2025) [{data['pct']:+.1f}%]")

Generated: ../paper/tables/trade_by_category.tex

LaTeX Table:

Summary by Category (High Relevance):
\begin{table}[htbp]
\centering
\caption{U.S. Import Values by AI Category, High AI Relevance (2023 vs 2025)}
\label{tab:trade_by_category}
\setlength{\tabcolsep}{4.5mm}
\renewcommand{\arraystretch}{1.60}
\begin{tabular}{lrrrr}
\toprule
Category & \# HS10 Codes & 2023 (\$B) & 2025 (\$B) & Change (\%) \\
\midrule
Compute Hardware & 163 & 144.4 & 353.8 & +144.9 \\
Electrical Power & 250 & 116.9 & 141.8 & +21.3 \\
Networking Telecom & 24 & 62.9 & 99.5 & +58.2 \\
Cooling HVAC & 137 & 41.5 & 47.5 & +14.4 \\
Building Structure & 44 & 12.1 & 10.1 & -16.5 \\
Fire Safety Security & 8 & 0.6 & 0.7 & +10.5 \\
Specialty Materials & 19 & 0.4 & 0.5 & +22.5 \\
\midrule
Total High AI Relevance & 645 & 379.0 & 654.0 & +72.6 \\
\bottomrule
\end{tabular}
\end{table}


Summary by Category (High Relevance):

  Total High AI Relevance: 645 HS10 codes | $379.0B (2023) → $654.0B (2025) [+72.6%]
  Specialty Mate

In [108]:
# Write trade by category values to LaTeX file
with open(texfile, 'a') as f:
    # Each category
    for cat in all_cats:
        data = category_data[cat]
        cat_clean = cat.replace('_', '')
        cat_name = cat.replace('_', ' ')
        f.write(f'\\newcommand{{\\{cat_clean}HsCodes}}{{{data["hs_count"]}}} % Number of HS10 codes in {cat_name}\n')
        f.write(f'\\newcommand{{\\{cat_clean}TwentyThree}}{{{data["2023"]/1e9:.0f}}} % {cat_name} imports in 2023 ($B)\n')
        f.write(f'\\newcommand{{\\{cat_clean}TwentyFive}}{{{data["2025"]/1e9:.0f}}} % {cat_name} imports in 2025 ($B)\n')
        f.write(f'\\newcommand{{\\{cat_clean}PctChange}}{{{data["pct"]:.0f}}} % {cat_name} percent change 2023-2025\n')
    
    # Total High AI Relevance
    f.write(f'\\def\\totalHighAiHsCodes{{{total_high_hs_count}}} % Total HS10 codes across all high AI categories\n')
    f.write(f'\\def\\totalHighAiTwentyThree{{{total_high_2023/1e9:.0f}}} % Total high AI imports in 2023 ($B)\n')
    f.write(f'\\def\\totalHighAiTwentyFive{{{total_high_2025/1e9:.0f}}} % Total high AI imports in 2025 ($B)\n')
    f.write(f'\\def\\totalHighAiPctChange{{{total_high_pct:.0f}}} % Total high AI percent change 2023-2025\n')

print(f"\nAppended trade by category values to {texfile}")


Appended trade by category values to ../paper/tables/ai-trade-results.tex


In [109]:
# Generate combined hierarchical LaTeX table for trade values
# Shows High AI Relevance with subcategories, then Low AI Relevance, then Total

import os
os.makedirs('paper/tables', exist_ok=True)

# Use the data already calculated in previous cells
# high_hs_count, high_2023, high_2025, high_pct (from relevance table)
# low_hs_count, low_2023, low_2025, low_pct (from relevance table)
# total_hs_count, total_2023, total_2025, total_pct (from relevance table)
# category_data, all_cats (from category table)

# Generate combined hierarchical table
table_file = "../paper/tables/trade_hierarchical.tex"
with open(table_file, 'w', encoding='utf-8') as f:
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\caption{U.S. Import Values by AI Relevance and Category (2023 vs 2025)}\n")
    f.write("\\label{tab:trade_hierarchical}\n")
    f.write("\\setlength{\\tabcolsep}{4.5mm}\n")
    f.write("\\renewcommand{\\arraystretch}{1.60}\n")
    f.write("{\\small\n")
    f.write("\\begin{tabular}{lcccc}\n")
    f.write("\\toprule\n")
    f.write("Category & \\# HS10 Codes & 2023 (\\$B) & 2025 (\\$B) & Change (\\%) \\\\\n")
    f.write("\\midrule\n")
    
    # High AI Relevance (main row)
    f.write(f"High AI Relevance & {high_hs_count} & {high_2023/1e9:.1f} & {high_2025/1e9:.1f} & {high_pct:+.1f} \\\\\n")
    
    # Subcategories (indented)
    for cat in all_cats:
        cat_name = cat.replace('_', ' ')
        data = category_data[cat]
        # Use \quad\quad for indentation
        f.write(f"\\quad\\quad {cat_name} & {data['hs_count']} & {data['2023']/1e9:.1f} & {data['2025']/1e9:.1f} & {data['pct']:+.1f} \\\\\n")
    
    # Low AI Relevance
    f.write(f"Low AI Relevance & {low_hs_count} & {low_2023/1e9:.1f} & {low_2025/1e9:.1f} & {low_pct:+.1f} \\\\\n")
    
    f.write("\\midrule\n")
    
    # Total Trade
    f.write(f"Total Trade & {total_hs_count} & {total_2023/1e9:.1f} & {total_2025/1e9:.1f} & {total_pct:+.1f} \\\\\n")
    
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}}\n")
    f.write("\\end{table}\n")

print(f"Generated: {table_file}\n")
print("LaTeX Table:")
print("="*80)
with open(table_file, 'r', encoding='utf-8') as f:
    print(f.read())
print("="*80)
print("\nSummary:")
print(f"High AI Relevance: {high_hs_count} HS10 codes | ${high_2023/1e9:.1f}B (2023) → ${high_2025/1e9:.1f}B (2025) [{high_pct:+.1f}%]")
for cat in all_cats:
    data = category_data[cat]
    cat_name = cat.replace('_', ' ')
    print(f"  └─ {cat_name}: {data['hs_count']} HS10 codes | ${data['2023']/1e9:.1f}B (2023) → ${data['2025']/1e9:.1f}B (2025) [{data['pct']:+.1f}%]")
print(f"Low AI Relevance: {low_hs_count} HS10 codes | ${low_2023/1e9:.1f}B (2023) → ${low_2025/1e9:.1f}B (2025) [{low_pct:+.1f}%]")
print(f"Total Trade: {total_hs_count} HS10 codes | ${total_2023/1e9:.1f}B (2023) → ${total_2025/1e9:.1f}B (2025) [{total_pct:+.1f}%]")

Generated: ../paper/tables/trade_hierarchical.tex

LaTeX Table:
\begin{table}[htbp]
\centering
\caption{U.S. Import Values by AI Relevance and Category (2023 vs 2025)}
\label{tab:trade_hierarchical}
\setlength{\tabcolsep}{4.5mm}
\renewcommand{\arraystretch}{1.60}
{\small
\begin{tabular}{lcccc}
\toprule
Category & \# HS10 Codes & 2023 (\$B) & 2025 (\$B) & Change (\%) \\
\midrule
High AI Relevance & 645 & 379.0 & 654.0 & +72.6 \\
\quad\quad Compute Hardware & 163 & 144.4 & 353.8 & +144.9 \\
\quad\quad Electrical Power & 250 & 116.9 & 141.8 & +21.3 \\
\quad\quad Networking Telecom & 24 & 62.9 & 99.5 & +58.2 \\
\quad\quad Cooling HVAC & 137 & 41.5 & 47.5 & +14.4 \\
\quad\quad Building Structure & 44 & 12.1 & 10.1 & -16.5 \\
\quad\quad Fire Safety Security & 8 & 0.6 & 0.7 & +10.5 \\
\quad\quad Specialty Materials & 19 & 0.4 & 0.5 & +22.5 \\
Low AI Relevance & 15915 & 1834.7 & 1880.9 & +2.5 \\
\midrule
Total Trade & 18364 & 2598.3 & 2883.9 & +11.0 \\
\bottomrule
\end{tabular}}
\end{table}




In [110]:
# Top 5 HS10 codes within each top category (High relevance only)

top_4_cats = ['Compute_Hardware', 'Electrical_Power', 'Networking_Telecom', 'Cooling_HVAC']

df_reset = df.reset_index()
df_high_only = df_reset[df_reset['relevance'] == 'High'].copy()

for cat in top_4_cats:
    print(f"\n{'='*80}")
    print(f"{cat.replace('_', ' ')}")
    print('='*80)
    
    # Filter to this category
    cat_data = df_high_only[df_high_only['primary_category'] == cat].copy()
    
    # Get total imports for this category
    cat_total = cat_data['imports'].sum()
    
    # Group by HS10 and sum imports
    hs10_imports = cat_data.groupby('HS10')['imports'].sum().sort_values(ascending=False)
    
    # Get top 5
    top_5_hs10 = hs10_imports.head(5)
    
    # Get descriptions from df and matlist
    for rank, (hs10, imports) in enumerate(top_5_hs10.items(), 1):
        fraction = (imports / cat_total) * 100
        
        # Get description from df and reasoning from matlist
        desc_row_df = cat_data[cat_data['HS10'] == hs10]
        desc_row_matlist = matlist[matlist['HS10'] == hs10]
        
        if len(desc_row_df) > 0:
            description = desc_row_df.iloc[0].get('short_description', 'No description available')
        else:
            description = 'Description not found'
            
        if len(desc_row_matlist) > 0:
            llm_reasoning = desc_row_matlist.iloc[0].get('reasoning', 'No reasoning available')
        else:
            llm_reasoning = 'Reasoning not found'

        
        print(f"   Reasoning: {llm_reasoning}")
        print(f"\n{rank}. HS10: {hs10}")        
        print(f"   Description: {description}")
        print(f"   Share of {cat.replace('_', ' ')}: {fraction:.2f}%")


Compute Hardware
   Reasoning: Digital processing units are core compute components used in data center servers. The NAICS classification confirms these are electronic computer components. These would include CPUs, GPUs, and other processors essential for AI training and inference operations in data centers.

1. HS10: 8471500150
   Description: PROC UNT IN HOUS W/ EITHER STOR, IN&OUTPUT,W/O CRT
   Share of Compute Hardware: 29.37%
   Reasoning: Electronic integrated circuits including processors and controllers are core components of AI data center compute infrastructure. These are essential for servers running AI workloads, whether CPUs for general processing or specialized processors for AI acceleration. The NAICS code 334413 confirms these are semiconductor devices, which are fundamental to data center operations.

2. HS10: 8542310001
   Description: PROCESSORS & CONTOLLERS W/NOT COMBO W MEMORIS, ETC
   Share of Compute Hardware: 8.75%
   Reasoning: This product covers parts and ac

In [111]:
# Generate LaTeX tables for top 5 HS10 codes in each category

import os

# Create output directory if it doesn't exist
os.makedirs('paper/tables', exist_ok=True)

def escape_latex(text):
    """Escape special LaTeX characters"""
    if pd.isna(text):
        return ''
    text = str(text)
    
    # First, escape backslash and special characters BEFORE adding math symbols
    text = text.replace('\\', r'\textbackslash{}')
    replacements = {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    
    # THEN replace Unicode math symbols with LaTeX equivalents (after $ is already escaped)
    unicode_math = {
        '≤': r'$\leq$',
        '≥': r'$\geq$',
        '≠': r'$\neq$',
        '≈': r'$\approx$',
        '×': r'$\times$',
        '÷': r'$\div$',
        '±': r'$\pm$',
        '°': r'$^\circ$',
        'μ': r'$\mu$',
        'Ω': r'$\Omega$',
        '→': r'$\rightarrow$',
        '←': r'$\leftarrow$',
    }
    for unicode_char, latex_cmd in unicode_math.items():
        text = text.replace(unicode_char, latex_cmd)
    
    return text

top_4_cats = ['Compute_Hardware', 'Electrical_Power', 'Networking_Telecom', 'Cooling_HVAC']

df_reset = df.reset_index()
df_high_only = df_reset[df_reset['relevance'] == 'High'].copy()

# Filter to 2023 and 2025 data
df_2023 = df_high_only[df_high_only['time'].dt.year == 2023].copy()
df_2025 = df_high_only[df_high_only['time'].dt.year == 2025].copy()

# Generate combined file with all 4 tables
combined_file = "../paper/tables/top5_all_categories.tex"
with open(combined_file, 'w', encoding='utf-8') as f:
    f.write("% Top 5 HS10 Codes for All AI Categories\n\n")
    
    for cat in top_4_cats:
        # Filter to this category for 2025 (to rank by 2025 imports)
        cat_data_2025 = df_2025[df_2025['primary_category'] == cat].copy()
        cat_data_2023 = df_2023[df_2023['primary_category'] == cat].copy()
        
        # Group by HS10 and sum imports for 2025 (for ranking)
        hs10_imports_2025 = cat_data_2025.groupby('HS10')['imports'].sum().sort_values(ascending=False)
        hs10_imports_2023 = cat_data_2023.groupby('HS10')['imports'].sum()
        
        # Get top 5 by 2025 imports
        top_5_hs10 = hs10_imports_2025.head(5)
        
        # Generate LaTeX table for this category
        f.write("\\begin{table}[htbp]\n")
        f.write("\\centering\n")
        f.write("\\small\n")
        f.write("\\caption{Top 5 HS10 Codes in " + cat.replace('_', ' ') + "}\n")
        f.write("\\label{tab:top5_" + cat.lower() + "}\n")
        f.write("\\setlength{\\tabcolsep}{1.25mm}\n")
        f.write("\\renewcommand{\\arraystretch}{1.80}\n")
        f.write("\\begin{tabular}{lp{3.5cm}p{3.5cm}cc}\n")
        f.write("\\toprule\n")
        f.write("HS10 Code & Description & LLM Reasoning & 2025 Imports (\\$B) & Change (\\%) \\\\\n")
        f.write("\\midrule\n")
        
        for rank, (hs10, imports_2025) in enumerate(top_5_hs10.items(), 1):
            imports_billions = imports_2025 / 1e9
            
            # Calculate percent change from 2023 to 2025
            imports_2023 = hs10_imports_2023.get(hs10, 0)
            if imports_2023 > 0:
                pct_change = ((imports_2025 - imports_2023) / imports_2023) * 100
            else:
                pct_change = 0
            
            # Get description from df and reasoning from matlist
            desc_row_df = cat_data_2025[cat_data_2025['HS10'] == hs10]
            desc_row_matlist = matlist[matlist['HS10'] == hs10]
            
            if len(desc_row_df) > 0:
                description = desc_row_df.iloc[0].get('short_description', 'No description available')
            else:
                description = 'Description not found'
                
            if len(desc_row_matlist) > 0:
                reasoning = desc_row_matlist.iloc[0].get('reasoning', 'No reasoning available')
            else:
                reasoning = 'Reasoning not found'
            
            # Extract only first sentence from reasoning
            reasoning = reasoning.split('.')[0] + '.' if '.' in reasoning else reasoning
            
            # Keep description and reasoning as-is without sentence case conversion
            
            # Escape LaTeX special characters
            description_escaped = escape_latex(description)
            reasoning_escaped = escape_latex(reasoning)
            
            f.write(f"{hs10} & {{\\footnotesize {description_escaped}}} & {{\\footnotesize {reasoning_escaped}}} & {imports_billions:.2f} & {pct_change:+.1f} \\\\\n")
        
        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write("\\end{table}\n\n")


print(f"Generated: {combined_file}")
print("\nRequired packages: \\usepackage{booktabs}")
print("  - \\input{tables/top5_all_categories.tex}")
print("\nTo use in your LaTeX document:")

Generated: ../paper/tables/top5_all_categories.tex

Required packages: \usepackage{booktabs}
  - \input{tables/top5_all_categories.tex}

To use in your LaTeX document:


In [112]:
# Generate LaTeX table for top 10 HS10 codes across all high relevance categories (by 2025 imports)

import os
os.makedirs('paper/tables', exist_ok=True)

df_reset = df.reset_index()
df_high_only = df_reset[df_reset['relevance'] == 'High'].copy()

# Filter to 2025 and 2023 data and group by HS10
df_2025 = df_high_only[df_high_only['time'].dt.year == 2025].copy()
df_2023 = df_high_only[df_high_only['time'].dt.year == 2023].copy()

hs10_imports_2025 = df_2025.groupby('HS10')['imports'].sum().sort_values(ascending=False)
hs10_imports_2023 = df_2023.groupby('HS10')['imports'].sum()

# Get top 10
top_10_hs10 = hs10_imports_2025.head(10)

# Calculate total trade imports in 2025 for percentage calculation
total_trade_2025 = df_reset[df_reset['time'].dt.year == 2025]['imports'].sum()

# Generate LaTeX table
table_file = "../paper/tables/top10_high_relevance.tex"
with open(table_file, 'w', encoding='utf-8') as f:
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\small\n")
    f.write("\\caption{Top 10 HS10 Codes by 2025 Import Volume, High AI Relevance}\n")
    f.write("\\label{tab:top10_high_relevance}\n")
    f.write("\\setlength{\\tabcolsep}{1.25mm}\n")
    f.write("\\renewcommand{\\arraystretch}{1.80}\n")
    f.write("\\begin{tabular}{lp{5cm}lccc}\n")
    f.write("\\toprule\n")
    f.write("HS10 Code & Description & Category & 2025 Imports (\\$B) & Share of Total Trade (\\%) & Change (\\%) " + r"\\")
    f.write("\n")
    f.write("\\midrule\n")

    for rank, (hs10, imports) in enumerate(top_10_hs10.items(), 1):
        fraction = (imports / total_trade_2025) * 100
        imports_billions = imports / 1e9

        # Calculate percent change from 2023 to 2025
        imports_2023 = hs10_imports_2023.get(hs10, 0)
        if imports_2023 > 0:
            pct_change = ((imports - imports_2023) / imports_2023) * 100
        else:
            pct_change = 0

        # Get description from df and category from matlist
        desc_row_df = df_2025[df_2025['HS10'] == hs10]
        desc_row_matlist = matlist[matlist['HS10'] == hs10]

        if len(desc_row_df) > 0:
            description = desc_row_df.iloc[0].get('short_description', 'No description available')
        else:
            description = 'Description not found'

        if len(desc_row_matlist) > 0:
            category = desc_row_matlist.iloc[0].get('primary_category', 'Unknown')
        else:
            category = 'Unknown'

        # Keep description as-is without sentence case conversion to preserve original formatting

        # Format category name
        category_formatted = category.replace('_', ' ')

        # Escape LaTeX special characters
        description_escaped = escape_latex(description)
        category_escaped = escape_latex(category_formatted)

        f.write(f"{hs10} & {{\\footnotesize {description_escaped}}} & {category_escaped} & {imports_billions:.2f} & {fraction:.2f} & {pct_change:+.1f} " + r"\\")
        f.write("\n")

    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")

print(f"Generated: {table_file}\n")
print("LaTeX Table:")
print("="*80)
with open(table_file, 'r', encoding='utf-8') as f:
    print(f.read())
print("="*80)
print("\nTop 10 HS10 Codes by 2025 Import Volume:")
for rank, (hs10, imports) in enumerate(top_10_hs10.items(), 1):
    fraction = (imports / total_trade_2025) * 100

    # Calculate percent change from 2023 to 2025
    imports_2023 = hs10_imports_2023.get(hs10, 0)
    if imports_2023 > 0:
        pct_change = ((imports - imports_2023) / imports_2023) * 100
    else:
        pct_change = 0

    desc_row_df = df_2025[df_2025['HS10'] == hs10]
    desc_row_matlist = matlist[matlist['HS10'] == hs10]

    if len(desc_row_df) > 0:
        description = desc_row_df.iloc[0].get('short_description', 'No description')
    else:
        description = 'Description not found'

    if len(desc_row_matlist) > 0:
        category = desc_row_matlist.iloc[0].get('primary_category', 'Unknown')
    else:
        category = 'Unknown'
    print(f"{rank}. HS10: {hs10} | {category.replace('_', ' ')} | ${imports/1e9:.2f}B | {fraction:.2f}% | Change: {pct_change:+.1f}% | {description[:60]}...")

Generated: ../paper/tables/top10_high_relevance.tex

LaTeX Table:
\begin{table}[htbp]
\centering
\small
\caption{Top 10 HS10 Codes by 2025 Import Volume, High AI Relevance}
\label{tab:top10_high_relevance}
\setlength{\tabcolsep}{1.25mm}
\renewcommand{\arraystretch}{1.80}
\begin{tabular}{lp{5cm}lccc}
\toprule
HS10 Code & Description & Category & 2025 Imports (\$B) & Share of Total Trade (\%) & Change (\%) \\
\midrule
8471500150 & {\footnotesize PROC UNT IN HOUS W/ EITHER STOR, IN\&OUTPUT,W/O CRT} & Compute Hardware & 163.55 & 5.67 & +343.0 \\
8517620090 & {\footnotesize MACH FOR RECP/CONVER ETC OF VOICE/IMAGE/DATA NESOI} & Networking Telecom & 52.88 & 1.83 & +58.3 \\
8473301180 & {\footnotesize PTS ADP MCH, NT INCPTNG CRT,PRT CRT ASSEM.;NESOI} & Compute Hardware & 48.89 & 1.70 & +199.8 \\
8517620020 & {\footnotesize SWITCHING AND ROUTING APPARATUS} & Networking Telecom & 30.76 & 1.07 & +89.1 \\
8473301140 & {\footnotesize PTS ADP MCH, NT INCPTG CRT,PRT CT ASM.,MRY MODULES} & Compute Har

In [113]:
# Generate LANDSCAPE LaTeX table for top 10 HS10 codes with LLM reasoning included

import os
os.makedirs('paper/tables', exist_ok=True)

# Recalculate the data to ensure we have the correct values
df_reset = df.reset_index()
df_high_only = df_reset[df_reset['relevance'] == 'High'].copy()

# Filter to 2025 and 2023 data and group by HS10
df_2025_landscape = df_high_only[df_high_only['time'].dt.year == 2025].copy()
df_2023_landscape = df_high_only[df_high_only['time'].dt.year == 2023].copy()

hs10_imports_2025_landscape = df_2025_landscape.groupby('HS10')['imports'].sum().sort_values(ascending=False)
hs10_imports_2023_landscape = df_2023_landscape.groupby('HS10')['imports'].sum()

# Get top 10
top_10_hs10_landscape = hs10_imports_2025_landscape.head(10)

# Calculate total trade imports in 2025 for percentage calculation
total_trade_2025_landscape = df_reset[df_reset['time'].dt.year == 2025]['imports'].sum()

# Generate LaTeX table with landscape format
table_file = "../paper/tables/top10_high_relevance_landscape.tex"
with open(table_file, 'w', encoding='utf-8') as f:
    f.write("\\begin{landscape}\n")
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\footnotesize\n")
    f.write("\\caption{Top 10 HS10 Codes by 2025 Import Volume with High AI Relevance}\n")
    f.write("\\label{tab:top10_high_relevance_landscape}\n")
    f.write("\\setlength{\\tabcolsep}{2mm}\n")
    f.write("\\renewcommand{\\arraystretch}{1.0}\n")
    f.write("\\begin{tabular}{lp{4cm}lp{6cm}ccc}\n")
    f.write("\\toprule\n")
    f.write("HS10 Code & Description & Category & LLM Reasoning & 2025 Imports (\\$B) & Share of Trade (\\%) & Change (\\%) " + r"\\")
    f.write("\n")
    f.write("\\midrule\n")

    for rank, (hs10, imports) in enumerate(top_10_hs10_landscape.items(), 1):
        fraction = (imports / total_trade_2025_landscape) * 100
        imports_billions = imports / 1e9

        # Calculate percent change from 2023 to 2025
        imports_2023 = hs10_imports_2023_landscape.get(hs10, 0)
        if imports_2023 > 0:
            pct_change = ((imports - imports_2023) / imports_2023) * 100
        else:
            pct_change = 0

        # Get description from df, category and reasoning from matlist
        desc_row_df = df_2025_landscape[df_2025_landscape['HS10'] == hs10]
        desc_row_matlist = matlist[matlist['HS10'] == hs10]

        if len(desc_row_df) > 0:
            description = desc_row_df.iloc[0].get('short_description', 'No description available')
        else:
            description = 'Description not found'

        if len(desc_row_matlist) > 0:
            category = desc_row_matlist.iloc[0].get('primary_category', 'Unknown')
            reasoning = desc_row_matlist.iloc[0].get('reasoning', 'No reasoning available')
        else:
            category = 'Unknown'
            reasoning = 'Reasoning not found'

        # Extract only first sentence from reasoning
        reasoning = reasoning.split('.')[0] + '.' if '.' in reasoning else reasoning

        # Format category name
        category_formatted = category.replace('_', ' ')

        # Escape LaTeX special characters
        description_escaped = escape_latex(description)
        category_escaped = escape_latex(category_formatted)
        reasoning_escaped = escape_latex(reasoning)

        f.write(f"{hs10} & {{\\scriptsize {description_escaped}}} & {category_escaped} & {{\\scriptsize {reasoning_escaped}}} & {imports_billions:.2f} & {fraction:.2f} & {pct_change:+.1f} " + r"\\")
        f.write("\n")

    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")
    f.write("\\end{landscape}\n")

print(f"Generated: {table_file}\n")
print("Required packages: \\usepackage{booktabs}, \\usepackage{lscape}")
print(f"  - \\input{{{table_file}}}")
print("\nLaTeX Table:")
print("="*80)
with open(table_file, 'r', encoding='utf-8') as f:
    print(f.read())
print("="*80)
print("\nTop 10 HS10 Codes with LLM Reasoning (Landscape Format):")
for rank, (hs10, imports) in enumerate(top_10_hs10_landscape.items(), 1):
    fraction = (imports / total_trade_2025_landscape) * 100
    imports_2023 = hs10_imports_2023_landscape.get(hs10, 0)
    if imports_2023 > 0:
        pct_change = ((imports - imports_2023) / imports_2023) * 100
    else:
        pct_change = 0

    desc_row_matlist = matlist[matlist['HS10'] == hs10]
    if len(desc_row_matlist) > 0:
        category = desc_row_matlist.iloc[0].get('primary_category', 'Unknown')
        reasoning = desc_row_matlist.iloc[0].get('reasoning', 'No reasoning available')
    else:
        category = 'Unknown'
        reasoning = 'Reasoning not found'

    reasoning_short = reasoning.split('.')[0] + '.' if '.' in reasoning else reasoning
    print(f"{rank}. HS10: {hs10} | {category.replace('_', ' ')} | ${imports/1e9:.2f}B (2025) | ${imports_2023/1e9:.2f}B (2023) | {fraction:.2f}% | Change: {pct_change:+.1f}%")
    print(f"   Reasoning: {reasoning_short}")

Generated: ../paper/tables/top10_high_relevance_landscape.tex

Required packages: \usepackage{booktabs}, \usepackage{lscape}
  - \input{../paper/tables/top10_high_relevance_landscape.tex}

LaTeX Table:
\begin{landscape}
\begin{table}[htbp]
\centering
\footnotesize
\caption{Top 10 HS10 Codes by 2025 Import Volume with High AI Relevance}
\label{tab:top10_high_relevance_landscape}
\setlength{\tabcolsep}{2mm}
\renewcommand{\arraystretch}{1.0}
\begin{tabular}{lp{4cm}lp{6cm}ccc}
\toprule
HS10 Code & Description & Category & LLM Reasoning & 2025 Imports (\$B) & Share of Trade (\%) & Change (\%) \\
\midrule
8471500150 & {\scriptsize PROC UNT IN HOUS W/ EITHER STOR, IN\&OUTPUT,W/O CRT} & Compute Hardware & {\scriptsize Digital processing units are core compute components used in data center servers.} & 163.55 & 5.67 & +343.0 \\
8517620090 & {\scriptsize MACH FOR RECP/CONVER ETC OF VOICE/IMAGE/DATA NESOI} & Networking Telecom & {\scriptsize This product covers data transmission and conversion eq

In [114]:
# Generate LANDSCAPE LaTeX table for HS10 codes ranked 11-20 with LLM reasoning

import os
os.makedirs('paper/tables', exist_ok=True)

# Reuse the data from previous cell (already calculated)
# Get products ranked 11-20
top_11to20_hs10_landscape = hs10_imports_2025_landscape.iloc[10:20]

# Total high-relevance imports in 2025 (for share % column)
total_high_2025_landscape = hs10_imports_2025_landscape.sum()

# Generate LaTeX table with landscape format
table_file = "../paper/tables/top11to20_high_relevance_landscape.tex"
with open(table_file, 'w', encoding='utf-8') as f:
    f.write("\\begin{landscape}\n")
    f.write("\\begin{table}[htbp]\n")
    f.write("\\centering\n")
    f.write("\\footnotesize\n")
    f.write("\\caption{HS10 Codes Ranked 11-20 by 2025 Import Volume with High AI Relevance}\n")
    f.write("\\label{tab:top11to20_high_relevance_landscape}\n")
    f.write("\\setlength{\\tabcolsep}{2mm}\n")
    f.write("\\renewcommand{\\arraystretch}{1.0}\n")
    f.write("\\begin{tabular}{lp{4cm}lp{6cm}ccc}\n")
    f.write("\\toprule\n")
    f.write("HS10 Code & Description & Category & LLM Reasoning & 2025 Imports (\\$B) & Share (\\%) & Change (\\%) \\\\\n")
    f.write("\\midrule\n")
    
    for rank, (hs10, imports) in enumerate(top_11to20_hs10_landscape.items(), 11):
        fraction = (imports / total_high_2025_landscape) * 100
        imports_billions = imports / 1e9
        
        # Calculate percent change from 2023 to 2025
        imports_2023 = hs10_imports_2023_landscape.get(hs10, 0)
        if imports_2023 > 0:
            pct_change = ((imports - imports_2023) / imports_2023) * 100
        else:
            pct_change = 0
        
        # Get description from df, category and reasoning from matlist
        desc_row_df = df_2025_landscape[df_2025_landscape['HS10'] == hs10]
        desc_row_matlist = matlist[matlist['HS10'] == hs10]
        
        if len(desc_row_df) > 0:
            description = desc_row_df.iloc[0].get('short_description', 'No description available')
        else:
            description = 'Description not found'
            
        if len(desc_row_matlist) > 0:
            category = desc_row_matlist.iloc[0].get('primary_category', 'Unknown')
            reasoning = desc_row_matlist.iloc[0].get('reasoning', 'No reasoning available')
        else:
            category = 'Unknown'
            reasoning = 'Reasoning not found'
        
        # Extract only first sentence from reasoning
        reasoning = reasoning.split('.')[0] + '.' if '.' in reasoning else reasoning
        
        # Format category name
        category_formatted = category.replace('_', ' ')
        
        # Escape LaTeX special characters
        description_escaped = escape_latex(description)
        category_escaped = escape_latex(category_formatted)
        reasoning_escaped = escape_latex(reasoning)
        
        f.write(f"{hs10} & {{\\scriptsize {description_escaped}}} & {category_escaped} & {{\\scriptsize {reasoning_escaped}}} & {imports_billions:.2f} & {fraction:.2f} & {pct_change:+.1f} \\\\\n")
    
    f.write("\\bottomrule\n")
    f.write("\\end{tabular}\n")
    f.write("\\end{table}\n")
    f.write("\\end{landscape}\n")

print(f"Generated: {table_file}\n")
print("Required packages: \\usepackage{booktabs}, \\usepackage{lscape}")
print(f"  - \\input{{{table_file}}}")
print("\nLaTeX Table:")
print("="*80)
with open(table_file, 'r', encoding='utf-8') as f:
    print(f.read())
print("="*80)
print("\nHS10 Codes Ranked 11-20 with LLM Reasoning (Landscape Format):")
for rank, (hs10, imports) in enumerate(top_11to20_hs10_landscape.items(), 11):
    fraction = (imports / total_high_2025_landscape) * 100
    imports_2023 = hs10_imports_2023_landscape.get(hs10, 0)
    if imports_2023 > 0:
        pct_change = ((imports - imports_2023) / imports_2023) * 100
    else:
        pct_change = 0
    
    desc_row_matlist = matlist[matlist['HS10'] == hs10]
    if len(desc_row_matlist) > 0:
        category = desc_row_matlist.iloc[0].get('primary_category', 'Unknown')
        reasoning = desc_row_matlist.iloc[0].get('reasoning', 'No reasoning available')
    else:
        category = 'Unknown'
        reasoning = 'Reasoning not found'
    
    reasoning_short = reasoning.split('.')[0] + '.' if '.' in reasoning else reasoning
    print(f"{rank}. HS10: {hs10} | {category.replace('_', ' ')} | ${imports/1e9:.2f}B (2025) | ${imports_2023/1e9:.2f}B (2023) | {fraction:.2f}% | Change: {pct_change:+.1f}%")
    print(f"   Reasoning: {reasoning_short}")

Generated: ../paper/tables/top11to20_high_relevance_landscape.tex

Required packages: \usepackage{booktabs}, \usepackage{lscape}
  - \input{../paper/tables/top11to20_high_relevance_landscape.tex}

LaTeX Table:
\begin{landscape}
\begin{table}[htbp]
\centering
\footnotesize
\caption{HS10 Codes Ranked 11-20 by 2025 Import Volume with High AI Relevance}
\label{tab:top11to20_high_relevance_landscape}
\setlength{\tabcolsep}{2mm}
\renewcommand{\arraystretch}{1.0}
\begin{tabular}{lp{4cm}lp{6cm}ccc}
\toprule
HS10 Code & Description & Category & LLM Reasoning & 2025 Imports (\$B) & Share (\%) & Change (\%) \\
\midrule
8542310045 & {\scriptsize PRCSSRS (INCL MICRO): CENTRL PROCSSNG UNITS (CPUS)} & Compute Hardware & {\scriptsize CPUs are essential compute hardware components in AI data centers, used in servers for both AI workloads and data center infrastructure management.} & 10.28 & 1.57 & +0.0 \\
8471704065 & {\scriptsize HARD DISK DRIVE UNT, NESOI, W/OUT EXTNL POWR SUPLY} & Compute Hardware &